In [1]:
# !pip install fusion_solar_py -q
# !pip -q install  fusion_solar_py pvlib retry-requests openmeteo_requests requests-cache 

In [2]:
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# FUSION_SOLAR_CLIENT_PASSWORD = user_secrets.get_secret("FUSION_SOLAR_CLIENT_PASSWORD")
# FUSION_SOLAR_CLIENT_USERNAME = user_secrets.get_secret("FUSION_SOLAR_CLIENT_USERNAME")
# LAT = float(user_secrets.get_secret("LAT"))
# LON = float(user_secrets.get_secret("LON"))

In [3]:
import os

from dotenv import load_dotenv

load_dotenv()
FUSION_SOLAR_CLIENT_PASSWORD = os.environ.get("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = os.environ.get("FUSION_SOLAR_CLIENT_USERNAME")
LAT = float(os.environ.get("LAT"))
LON = float(os.environ.get("LON"))

In [4]:
import logging
import gymnasium as gym
from gymnasium import spaces
import numpy as np

from energymanagementrl.fusion_solar_connector import *
from energymanagementrl.production_forecast import *
from energymanagementrl.rl import extract_values_gen, EnergyManagementSystem, load_model_with_weights

In [5]:
# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Create a file handler for logging to a file
file_handler = logging.FileHandler('.log')
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Create a console handler for logging to the screen
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Add both handlers to the logger

for handler in logger.handlers[:]:
    logger.removeHandler(handler)
logger.addHandler(file_handler)
logger.addHandler(console_handler)



In [6]:
panel_model = PanelModel(pdc0=0.42, temp_model_a=-3.56, temp_model_b=-0.075, delta_t=3, gamma_pdc=-0.004)
num_panels = 14
arrays = [ArrayConfig(name='sud_east', panel_model=panel_model, num_panels=num_panels, tilt_angle=25, azimuth=110),
          ArrayConfig(name='nord_west', panel_model=panel_model, num_panels=num_panels, tilt_angle=18, azimuth=290)]

_plant_config = PlantConfig(
    latitude=LAT, longitude=LON, timezone='Europe/Rome', inverter_pdc0=6, arrays=arrays
)
_production_forecaster = EnergyPredictionSystem(plant_config=_plant_config, open_meteo_client=OpenMeteoClient())

In [7]:
_client = FusionSolarClientParsed(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                  huawei_subdomain="uni004eu5")
periodic_task = PeriodicTask(_client.keep_alive)
periodic_task.start()
_plant_id = _client.get_plant_ids()[0]
battery_id = _client.get_battery_ids(_plant_id)[0]

Periodic task started.


In [12]:
# Create a simple dummy environment used only to define the model struct, as the model will be used in inference
env = gym.Env()
env.action_space = spaces.Discrete(2)  # Two possible actions: 0 or 1
env.observation_space = spaces.Box(low=0, high=1000, shape=(55,), dtype=np.float64)  # 55 state variables
# Load the DQN policy
_model=load_model_with_weights(env,
                               '../data/trained_models/models/'
                               'dqn_1.0_0.06_0.06_0.02_1000_l_2.policy_weights.pth',
                               # 'dqn.2024_12_31_11_44_29.latest'
                               )

In [13]:
system = EnergyManagementSystem(
    client=_client,
    plant_id=_plant_id,
    battery_id=battery_id,
    production_forecaster=_production_forecaster,
    model=_model,
)

In [14]:
system.control_loop(active=False)

2025-01-01 16:02:29,315 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2025-01-01 16:02:32,737 - WARNING - Control loop terminated, battery mode reset.


KeyboardInterrupt: 

In [ ]:
state = system.get_system_state()
obs = np.array(list(extract_values_gen(state)))
# obs[-3]=200
# obs[-4]=0
# obs[-2]=200
# obs[-5] = 8800

if len(obs) == 55:
    action, _ = _model.predict(obs / 1000)
    print(int(action))
state